In [4]:
import pandas as pd
import numpy as np
import random
import copy

In [ ]:
from typing import Callable

def mse(y_true: pd.Series, y_pred: pd.Series):
    squared_differences = np.square(y_true - y_pred)
    mse = np.mean(squared_differences)
    return mse

def mae(y_true: pd.Series, y_pred: pd.Series):
    absolute_differrences = np.abs(y_true - y_pred)
    mae = np.mean(absolute_differrences)
    return mae

def rmse(y_true: pd.Series, y_pred: pd.Series):
    squared_differences = np.square(y_true - y_pred)
    mse = np.mean(squared_differences)
    rmse = np.sqrt(mse)
    return rmse

def r2(y_true: pd.Series, y_pred: pd.Series):
    y_mean = np.mean(y_true)
    squared_differences = np.sum(np.square(y_true - y_pred))
    squared_differences_mean = np.sum(np.square(y_true - y_mean))
    r2 = 1 - squared_differences / squared_differences_mean
    return r2


def mape(y_true: pd.Series, y_pred: pd.Series):
    mape = 100 * np.mean(np.abs(1 - y_pred / y_true))
    return mape



class MyLineReg():
    def __init__(self, n_iter=100, learning_rate=0.1, metric=None,
                 reg=None, l1_coef=0, l2_coef=0,
                 sgd_sample=None, random_state=42):
        self.n_iter = n_iter
        self.learning_rate = learning_rate
        self.weights = None
        self.metric = metric
        self.reg = reg
        self.l1_coef = l1_coef
        self.l2_coef = l2_coef
        self.sgd_sample = sgd_sample
        self._seed(random_state)
        self.__metric_value = None

    def _seed(self, random_state):
        self.random_state = random_state 
        random.seed(self.random_state)
        
    def __str__(self):
        return f"MyLineReg class: n_iter={self.n_iter}, learning_rate={self.learning_rate}"

    @staticmethod
    def _mse(y_true: pd.Series, y_pred: pd.Series):
        squared_differences = np.square(y_true - y_pred)
        mse = np.mean(squared_differences)
        return mse

    @staticmethod
    def _mae(y_true: pd.Series, y_pred: pd.Series):
        absolute_differrences = np.abs(y_true - y_pred)
        mae = np.mean(absolute_differrences)
        return mae

    @staticmethod
    def _rmse(y_true: pd.Series, y_pred: pd.Series):
        squared_differences = np.square(y_true - y_pred)
        mse = np.mean(squared_differences)
        rmse = np.sqrt(mse)
        return rmse

    @staticmethod
    def _r2(y_true: pd.Series, y_pred: pd.Series):
        y_mean = np.mean(y_true)
        squared_differences = np.sum(np.square(y_true - y_pred))
        squared_differences_mean = np.sum(np.square(y_true - y_mean))
        r2 = 1 - squared_differences / squared_differences_mean
        return r2

    @staticmethod
    def _mape(y_true: pd.Series, y_pred: pd.Series):
        mape = 100 * np.mean(np.abs(1 - y_pred / y_true))
        return mape

    def _get_metric_value(self, y_true, y_pred):
        if self.metric == 'mae':
            return self._mae(y_true, y_pred)
        if self.metric == 'mse':
            return self._mse(y_true, y_pred)
        if self.metric == 'rmse':
            return self._rmse(y_true, y_pred)
        if self.metric == 'mape':
            return self._mape(y_true, y_pred)
        if self.metric == 'r2':
            return self._r2(y_true, y_pred)
    
    def fit(self, X: pd.DataFrame, y: pd.Series, verbose=False):
        
        if '__bias' in X.columns:
            raise Exception("Column '__bias' exists in X")
        
        X = X.copy()

        X.insert(loc=0, column='__bias', value=1)

        count_features = len(X.columns)
        n = len(X)
            
        self.weights = pd.Series(1, X.columns)

        for i in range(1, self.n_iter + 1):
            
            y_pred = X.dot(self.weights)
            loss = self._mse(y, y_pred)

            if self.reg == 'l1':
                loss += self.l1_coef * np.sum(np.abs(self.weights))

            if self.reg == 'l2':
                loss += self.l2_coef * np.sum(np.square(self.weights))

            if self.reg == 'elasticnet':
                loss += self.l1_coef * np.sum(np.abs(self.weights)) + self.l2_coef * np.sum(np.square(self.weights))
            
            if i == 1 and verbose:  
                if self.metric is None:
                    print(f"start | loss: {loss}")
                else:
                    print(f"start | loss: {loss} | {self.metric}: {self._get_metric_value(y, y_pred)}")
            
            if self.sgd_sample is not None:
                if isinstance(self.sgd_sample, int):
                    sgd_sample = self.sgd_sample
                elif isinstance(self.sgd_sample, float):
                    sgd_sample = round(self.sgd_sample * n)
                    
                sample_rows_idx = random.sample(range(X.shape[0]), sgd_sample)
            
                X_hat = []
                y_hat = []
                y_pred_hat = []
                indexes_hat = []
                for j in sample_rows_idx:
                    X_hat.append(X.iloc[j])
                    y_hat.append(y.iloc[j])
                    y_pred_hat.append(y_pred.iloc[j])
                    indexes_hat.append(X.index[j])
                X_hat = pd.concat(X_hat, axis=1).T
                y_hat = pd.Series(y_hat, index= indexes_hat)
                y_pred_hat = pd.Series(y_pred_hat, index= indexes_hat)
            else:
                X_hat = X
                y_hat = y
                y_pred_hat = y_pred
            
            gradient = 2 * (y_pred_hat - y_hat).dot(X_hat) / len(X_hat)

            if self.reg == 'l1':
                gradient += self.l1_coef * np.sign(self.weights)

            if self.reg == 'l2':
                gradient += self.l2_coef * self.weights * 2

            if self.reg == 'elasticnet':
                gradient += self.l1_coef * np.sign(self.weights) + self.l2_coef * self.weights * 2            
            
            if isinstance(self.learning_rate, float):
                learning_rate = self.learning_rate
            
            if isinstance(self.learning_rate, Callable):
                learning_rate = self.learning_rate(i)
            
            self.weights -= learning_rate * gradient

            if verbose and (i) % verbose == 0 and i > 1:  
                if self.metric is None:
                    print(f"{i} | loss: {loss}")
                else:
                    print(f"{i} | loss: {loss} | {self.metric}: {self._get_metric_value(y, y_pred)}")

        y_pred = X.dot(self.weights)
        self.__metric_value = self._get_metric_value(y, y_pred)

    def get_best_score(self):
        return self.__metric_value

    def get_coef(self) -> np.ndarray:
        if self.weights is None:
            return None

        return self.weights.values[1:]

    def predict(self, X: pd.DataFrame) -> pd.Series :
        X = X.copy()
        X.insert(loc=0, column='__bias', value=1)
        return X.dot(self.weights)



def get_mse(y: pd.Series):
    return ((y - y.mean()) ** 2).mean()

 
class Leaf:
    def __init__(self, parent, samples, y, depth):
        self.parent = parent
        self.samples = samples
        self.y = y
        self.depth = depth

    def __str__(self):
        return f'{"  " * self.depth} {self.proba}'

class Node:
    def __init__(self, parent=None):
        self.parent = parent
        self.left = None
        self.right = None
        self.samples = None
        self.depth = 1
        self.split_value = None
        self.column = None
        self.ig = None
        self.is_left = None

    def __str__(self):
        return  '\n'.join([
        f'{"  " * self.depth} {self.column} {self.split_value}',
        f'{self.left}',
        f'{self.right}'])

    def predict(self, row: pd.Series):
        if row.loc[self.column] <= self.split_value:
            if isinstance(self.left, Leaf):
                return self.left.y
            else: 
                return self.left.predict(row)
        else:
            if isinstance(self.right, Leaf):
                return self.right.y
            else: 
                return self.right.predict(row)

class MyTreeReg:
    def __init__(self, max_depth: int = 5, min_samples_split: int = 2,
                 max_leafs: int = 20, bins=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_leafs = max_leafs
        self.nodes_cnt = 1
        self.leafs_cnt = None
        self.bins = bins
        self.split_values = None
        self.fi = dict()
        self.N = None

    def __str__(self):
        return f"MyTreeReg class: max_depth={self.max_depth}, min_samples_split={self.min_samples_split}, max_leafs={self.max_leafs}"

    def __build_split_values(self, X):
        self.split_values = dict()
        for column in X.columns:
            unique_values = X[column].unique()
            if (len(unique_values) - 1 <= self.bins - 1):
                sorted_unique_values = np.sort(unique_values)
                self.split_values[column] = (sorted_unique_values[:-1] + sorted_unique_values[1:]) / 2
            else:
                count_elements, split_values = np.histogram(X[column], bins = self.bins)
                self.split_values[column] = split_values[1:-1]

    def predict(self, X: pd.DataFrame):
        y = []
        for index, row in X.iterrows():
            y_index = self.root.predict(row)
            y.append(y_index)
        return pd.Series(y, X.index)

    def get_split_values(self, X, column):
        if self.bins is None:
            unique_values = X[column].unique()
            sorted_unique_values = np.sort(unique_values)
            return (sorted_unique_values[:-1] + sorted_unique_values[1:]) / 2
        else:
            max_value = X[column].max()
            min_value = X[column].min()
            return self.split_values[column][ (min_value <= self.split_values[column]) & (max_value > self.split_values[column])]

    def __init_fi(self, X):
        for column in X.columns:
            self.fi[column] = 0 
    
    def fit(self, X: pd.DataFrame, y: pd.Series):
        self.root = Node()
        self.__init_fi(X)
        self.N = len(y)
        if self.bins is not None:
            self.__build_split_values(X)
            
        self.build_tree_recursive(self.root, X, y)
        self.leafs_cnt = self.nodes_cnt + 1

    def get_best_split(self, X: pd. DataFrame, y : pd.Series):

        igs = []
        
        for column in X.columns:
            split_values = self.get_split_values(X, column)
            criterion_function = get_mse
            criterion = criterion_function(y)
            
            for split_value in split_values: 

                left_indexes = X.index[X[column] <= split_value]
                left_y = y.loc[left_indexes]
                left_criterion = criterion_function(left_y)
                
                right_indexes = X.index[X[column] > split_value]
                right_y = y.loc[right_indexes]
                right_criterion = criterion_function(right_y)
    
                ig = criterion - left_criterion * len(left_indexes) / len(y) - right_criterion * len(right_indexes) / len(y) 
    
                igs.append((column, split_value, ig))
        if igs:    
            return max(igs, key= lambda x: x[2])
        else:
            return None, None, None

        
    def build_tree_recursive(self, node: Node, X, y):

        node.column, node.split_value, node.ig = self.get_best_split(X, y)
        
        if node.column is not None:
            self.fi[node.column] += len(y) / self.N * node.ig
            
            left_indexes = X.index[X[node.column] <= node.split_value]
            left_cnt = len(left_indexes)
            left_y = y.loc[left_indexes]
            
            if len(left_y.unique()) == 1 or left_cnt < self.min_samples_split or \
                self.nodes_cnt + 1 >= self.max_leafs or node.depth == self.max_depth \
                :
                node.left = Leaf(node, len(left_y), left_y.mean(), node.depth + 1)
            else:
                node.left = Node(node)
                node.left.depth = node.depth + 1
                node.left.is_left = True
                self.nodes_cnt += 1
                self.build_tree_recursive(node.left, X.loc[left_indexes], left_y)
    
            right_indexes = X.index[X[node.column] > node.split_value]
            right_cnt = len(right_indexes)
            right_y = y.loc[right_indexes]
            
            if len(right_y.unique()) == 1  or right_cnt < self.min_samples_split  or \
                self.nodes_cnt + 1 >= self.max_leafs or node.depth == self.max_depth:
                node.right = Leaf(node, len(right_y), right_y.mean(), node.depth + 1)
            else:
                node.right = Node(node)
                node.right.depth = node.depth + 1
                node.right.is_left = False
                self.nodes_cnt += 1
                self.build_tree_recursive(node.right, X.loc[right_indexes], right_y)
        else:
            parent = node.parent
            if node.is_left:
                parent.left = Leaf(parent, len(y), y.mean(), parent.depth + 1)
            else:
                parent.right = Leaf(parent, len(y), y.mean(), parent.depth + 1)
            self.nodes_cnt -= 1

class MyKNNReg:
    def __init__(self, k = 3, metric = 'euclidean', weight = 'uniform'):
        self.k = k 
        self.train_size = None
        self.metric = metric
        self.weight = weight

    def __str__(self):
        return f"MyKNNReg class: k={self.k}"

    def fit(self, X: pd.DataFrame, y: pd.Series):
        self.X = X.copy()
        self.y = y.copy()
        self.train_size = self.X.shape

    def _get_distances(self, x: pd.Series):
        if self.metric == 'euclidean':
            return np.sqrt(((self.X - x) ** 2).sum(axis=1))
        if self.metric == 'chebyshev':
            return (self.X - x).abs().max(axis=1)
        if self.metric == 'manhattan':
            return (self.X - x).abs().sum(axis=1)
        if self.metric == 'cosine':
            return 1 - (self.X * x).sum(axis=1) / ( np.sqrt((self.X ** 2).sum(axis=1)) * ((x ** 2).sum() ** 0.5) )
    
    def predict(self, X: pd.DataFrame):
        y_pred = []
        for index, row in X.iterrows():
            distances = self._get_distances(row)
            distances.sort_values(ascending=True, inplace=True)
            train_indexes = distances.index[:self.k]
            k_nearest_neighbor_distances = distances.loc[train_indexes]
            k_nearest_neighbor_y = self.y.loc[train_indexes]
            if self.weight == 'uniform':
                y_hat = k_nearest_neighbor_y.mean()
            if self.weight == 'rank':
                k_nearest_neighbor_rank = pd.Series(range(1, self.k + 1), train_indexes)
                sum_weight = (1 / k_nearest_neighbor_rank).sum()
                y_hat = (k_nearest_neighbor_y / k_nearest_neighbor_rank).sum() / sum_weight
            if self.weight == 'distance':
                sum_weight = (1 / k_nearest_neighbor_distances).sum()
                y_hat = (k_nearest_neighbor_y / k_nearest_neighbor_distances).sum() / sum_weight
            y_pred.append(y_hat)
        return pd.Series(y_pred, X.index)


class MyBaggingReg:
    def __init__(self, estimator = None , n_estimators = 10, max_samples = 1.0, random_state = 42, oob_score=None):
        self.estimator = estimator
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.random_state = random_state
        self.estimators = []
        self.oob_score = oob_score
        self.oob_score_ = None

    def __str__(self):
        return f"MyBaggingReg class: estimator={self.estimator}, n_estimators={self.n_estimators}, max_samples={self.max_samples}, random_state={self.random_state}"

    def fit(self, X: pd.DataFrame, y: pd.Series):
        n = round(self.max_samples * len(X))
        random.seed(self.random_state)
        X_s = []
        y_s = []
        if self.oob_score is not None:
            oob_X_s = []
            oob_y_s = []
            oob_y_pred_s = []
        for i in range(self.n_estimators):
            sample_rows_idx = random.choices(X.index.to_list(), k=n)
            X_hat = X.loc[sample_rows_idx]
            y_hat = y.loc[sample_rows_idx]
            index = pd.Index(range(len(X_hat)))
            X_hat.index = index
            y_hat.index = index
            X_s.append(X_hat)
            y_s.append(y_hat)

            if self.oob_score is not None:
                set_sample_rows_idx = set(sample_rows_idx)
                oob_indexes = X.index[X.index.map(lambda x: x not in set_sample_rows_idx)]
                oob_X = X.loc[oob_indexes]
                oob_y = y.loc[oob_indexes]
                oob_X_s.append(oob_X)
                oob_y_s.append(oob_y)
                

        for i in range(self.n_estimators):
            estimator = copy.deepcopy(self.estimator)
            estimator.fit(X_s[i], y_s[i])
            self.estimators.append(estimator)

            if self.oob_score is not None:
                oob_y_pred = estimator.predict(oob_X_s[i])
                oob_y_pred_s.append(oob_y_pred)

        if self.oob_score is not None:
            oob_y_pred = pd.concat(oob_y_pred_s, axis=1).mean(axis=1)
            list_index_oob = oob_y_pred.index.to_list()
            oob_y_true = y.loc[list_index_oob]

            if self.oob_score == "mae":
                metric = mae
                
            if self.oob_score == "mse":
                metric = mse
                
            if self.oob_score == "rmse":
                metric = rmse
                
            if self.oob_score == "mape":
                metric = mape
                
            if self.oob_score == "r2":
                metric = r2

            self.oob_score_ = metric(oob_y_true, oob_y_pred)
                
    def predict(self, X: pd.DataFrame):
        y_pred_s = []
        for estimator in self.estimators:
            y_pred = estimator.predict(X)
            y_pred_s.append(y_pred)
        y_pred = pd.concat(y_pred_s, axis=1)
        return y_pred.mean(axis=1)
        

In [8]:
pd.Index(range(5))

RangeIndex(start=0, stop=5, step=1)

In [2]:
X = pd.DataFrame({"a": [1,2,3]})

In [10]:
X.loc[[1,1,1,]].index.loc[[1,1]]

AttributeError: 'Index' object has no attribute 'loc'